In [3]:
!pip install yfinance -q

In [4]:
# Run in Google Colab: first cell  ->  !pip install yfinance -q
# Second cell: paste this script. It creates /content/Popular_Stocks_Sample.csv
import pandas as pd
import yfinance as yf

# ticker: (company, sector, is_etf)
TICKERS = {
    "NVDA": ("Nvidia", "Semiconductors", False),
    "TSLA": ("Tesla", "Autos", False),
    "AAPL": ("Apple", "Consumer Tech", False),
    "AMZN": ("Amazon", "Retail / Cloud", False),
    "MSFT": ("Microsoft", "Software", False),
    "GOOGL": ("Alphabet", "Communication", False),
    "META": ("Meta", "Communication", False),
    "PLTR": ("Palantir", "Software", False),
    "NFLX": ("Netflix", "Media", False),
    "F": ("Ford", "Autos", False),
    "VOO": ("Vanguard S&P 500 ETF", "Index", True),
}

raw = yf.download(list(TICKERS), period="5y", auto_adjust=True, progress=False)

# wide -> long: one row per ticker per day
df = (raw.stack(level=1, future_stack=True)
         .rename_axis(["Date", "Ticker"])
         .reset_index())
df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

df["Company"] = df["Ticker"].map(lambda t: TICKERS[t][0])
df["Sector"] = df["Ticker"].map(lambda t: TICKERS[t][1])
df["Is_ETF"] = df["Ticker"].map(lambda t: TICKERS[t][2])

df["Daily_Return"] = df.groupby("Ticker")["Close"].pct_change() * 100  # in %
df["Up_Day"] = df["Daily_Return"] > 0
df.loc[df["Daily_Return"].isna(), "Up_Day"] = pd.NA  # first day has no return

df = df[["Date", "Ticker", "Company", "Sector", "Is_ETF",
         "Open", "High", "Low", "Close", "Volume", "Daily_Return", "Up_Day"]]

df.to_csv("/content/Popular_Stocks_Sample.csv", index=False)
print(df.shape)
print(df.head())

/tmp/ipykernel_1574/695295048.py:35: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'nan' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df.loc[df["Daily_Return"].isna(), "Up_Day"] = pd.NA  # first day has no return


(13794, 12)
Price       Date Ticker Company         Sector  Is_ETF        Open  \
0     2021-11-19   AAPL   Apple  Consumer Tech   False  153.874774   
1     2021-11-22   AAPL   Apple  Consumer Tech   False  157.808200   
2     2021-11-23   AAPL   Apple  Consumer Tech   False  157.261660   
3     2021-11-24   AAPL   Apple  Consumer Tech   False  156.900491   
4     2021-11-26   AAPL   Apple  Consumer Tech   False  155.748786   

Price        High         Low       Close     Volume  Daily_Return Up_Day  
0      157.164084  152.781600  156.705338  117305600           NaN    NaN  
1      161.731936  157.144491  157.164017  117467900      0.292702   True  
2      157.925384  155.250993  157.544724   96041900      0.242235   True  
3      158.257204  155.817072  158.061996   69463600      0.328334   True  
4      156.607703  152.615650  153.054871   76959800     -3.167824  False  


# EXCERCISE
# You are about to put money into 11 popular US stocks and index funds, and you want to know what you would be signing up for before you do. Work in Colab with Popular_Stocks_Sample.csv: one row per ticker per trading day, about 5 years of data.


# Before any maths, confirm that the file contains what you think it contains.
# Load the file into df. How many rows and columns does it have, and what does one row represent? Answer in one sentence.


In [5]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv('/content/Popular_Stocks_Sample.csv')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13794 entries, 0 to 13793
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          13794 non-null  object 
 1   Ticker        13794 non-null  object 
 2   Company       13794 non-null  object 
 3   Sector        13794 non-null  object 
 4   Is_ETF        13794 non-null  bool   
 5   Open          13794 non-null  float64
 6   High          13794 non-null  float64
 7   Low           13794 non-null  float64
 8   Close         13794 non-null  float64
 9   Volume        13794 non-null  int64  
 10  Daily_Return  13783 non-null  float64
 11  Up_Day        13783 non-null  object 
dtypes: bool(1), float64(5), int64(1), object(5)
memory usage: 1.2+ MB


In [6]:
df.shape

(13794, 12)

In [7]:
# I want to see what columns have empty values. I can see that only two columns:
# Daily_Return and Up_Day
df.isna().sum()

,0
Date,0
Ticker,0
Company,0
Sector,0
Is_ETF,0
Open,0
High,0
Low,0
Close,0
Volume,0


In [8]:
# I am using double [] df[df[]] because the internal is the mask the external
# applies the mask.
# So i am visualizing the NaN # I tried to do sum() but that means nothing.
# I am seeing just the 11 rows that contain the NaN
df[df['Daily_Return'].isna()]

,Date,Ticker,Company,Sector,Is_ETF,Open,High,Low,Close,Volume,Daily_Return,Up_Day
0,2021-11-19,AAPL,Apple,Consumer Tech,False,153.874774,157.164084,152.781600,156.705338,117305600,NaN,NaN
1254,2021-11-19,AMZN,Amazon,Retail / Cloud,False,185.634506,188.107498,183.785995,183.828506,98734000,NaN,NaN
2508,2021-11-19,F,Ford,Autos,False,14.301811,14.323657,14.010532,14.119761,64610400,NaN,NaN
3762,2021-11-19,GOOGL,Alphabet,Communication,False,148.561813,149.543484,147.495468,147.522705,33682000,NaN,NaN
5016,2021-11-19,META,Meta,Communication,False,338.953603,348.759677,336.675405,342.024170,26488500,NaN,NaN
6270,2021-11-19,MSFT,Microsoft,Software,False,329.442601,331.807842,329.019546,329.894470,21963400,NaN,NaN
7524,2021-11-19,NFLX,Netflix,Media,False,69.235001,69.416000,67.500000,67.879997,26137000,NaN,NaN
8778,2021-11-19,NVDA,Nvidia,Semiconductors,False,32.101604,32.942900,31.765087,32.840351,533867000,NaN,NaN
10032,2021-11-19,PLTR,Palantir,Software,False,21.600000,21.959999,21.299999,21.410000,30013100,NaN,NaN
11286,2021-11-19,TSLA,Tesla,Autos,False,366.290009,379.573334,364.233337,379.019989,64926900,NaN,NaN


In [ ]:
What to do with them:

For return statistics (mean, median, std, correlation), pandas skips
NaN automatically, so you don’t need to do anything.
For the mask df[df['Up_Day']], those 11 empty values will cause
the same ValueError I hit with Licensed.
Use df[df['Up_Day'] == True] or drop the 11 rows first:

python
df = df.dropna(subset=['Daily_Return'])

In [9]:
# I AM ELIMINATING the NAN from Daily Return
df = df.dropna(subset=['Daily_Return'])

In [10]:
# I can see that here there are no NaN anymore
df[df['Daily_Return'].isna()]

,Date,Ticker,Company,Sector,Is_ETF,Open,High,Low,Close,Volume,Daily_Return,Up_Day


Sorting out which missing-value markers the checklist should cover.
Sorting out which missing-value markers the checklist should cover.
NaN and null are the same thing in pandas (also None and NaT for dates). isna() catches all of them, so that’s one item, not two. A data check has more items:

What to look for	Why it matters	Command

Missing values. |	Skew stats or break masks	| df.isna().sum()

Hidden missing values ('', 'N/A', '-', 'unknown')	| isna() doesn’t see them | df['col'].value_counts() and look for odd entries
Duplicate rows	| Count things twice |	df.duplicated().sum()
Wrong data types (numbers stored as text, dates as object)	| Calculations fail or sort wrongly	| df.info()
Impossible values (negative price, 0 views, a date in the future)	| Usually errors	| df.describe(), check min and max
Placeholder zeros (0 used for “unknown”)	| Pull mean and median down	(df['col'] == 0).sum()
Outliers	Distort mean and std	Compare mean with median, check max, IQR
Inconsistent categories (Apple, apple, Apple)	Count as separate values	df['col'].unique(), nunique()
Row count versus expectation	Reveals missing chunks	df.shape, groupby().size()
Imbalance (one category dominates)	Makes averages misleading	value_counts(normalize=True)Sorting out which missing-value markers the checklist should cover.
Sorting out which missing-value markers the checklist should cover.
NaN and null are the same thing in pandas (also None and NaT for dates). isna() catches all of them, so that’s one item, not two. A data check has more items:



In [11]:
df.head()

,Date,Ticker,Company,Sector,Is_ETF,Open,High,Low,Close,Volume,Daily_Return,Up_Day
1,2021-11-22,AAPL,Apple,Consumer Tech,False,157.808200,161.731936,157.144491,157.164017,117467900,0.292702,True
2,2021-11-23,AAPL,Apple,Consumer Tech,False,157.261660,157.925384,155.250993,157.544724,96041900,0.242235,True
3,2021-11-24,AAPL,Apple,Consumer Tech,False,156.900491,158.257204,155.817072,158.061996,69463600,0.328334,True
4,2021-11-26,AAPL,Apple,Consumer Tech,False,155.748786,156.607703,152.615650,153.054871,76959800,-3.167824,False
5,2021-11-29,AAPL,Apple,Consumer Tech,False,155.553564,157.329988,154.987452,156.402740,88748200,2.187366,True


In [12]:
df.tail()

,Date,Ticker,Company,Sector,Is_ETF,Open,High,Low,Close,Volume,Daily_Return,Up_Day
13789,2026-11-12,VOO,Vanguard S&P 500 ETF,Index,True,693.059998,701.419983,692.840027,696.000000,6184800,-0.336504,False
13790,2026-11-13,VOO,Vanguard S&P 500 ETF,Index,True,698.659973,698.809998,687.500000,693.380005,5626200,-0.376436,False
13791,2026-11-16,VOO,Vanguard S&P 500 ETF,Index,True,689.109985,689.219971,678.469971,680.260010,6706700,-1.892180,False
13792,2026-11-17,VOO,Vanguard S&P 500 ETF,Index,True,678.820007,679.979980,668.270020,673.190002,10039700,-1.039310,False
13793,2026-11-18,VOO,Vanguard S&P 500 ETF,Index,True,674.109985,675.679993,668.059998,671.309998,9912300,-0.279268,False


# Run df.info(). Which columns have missing values, and how many each? Which columns are object and which are float64? Date will probably be object: convert it with pd.to_datetime and check again.


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13783 entries, 1 to 13793
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Date          13783 non-null  object 
 1   Ticker        13783 non-null  object 
 2   Company       13783 non-null  object 
 3   Sector        13783 non-null  object 
 4   Is_ETF        13783 non-null  bool   
 5   Open          13783 non-null  float64
 6   High          13783 non-null  float64
 7   Low           13783 non-null  float64
 8   Close         13783 non-null  float64
 9   Volume        13783 non-null  int64  
 10  Daily_Return  13783 non-null  float64
 11  Up_Day        13783 non-null  object 
dtypes: bool(1), float64(5), int64(1), object(5)
memory usage: 1.8+ MB


In [13]:
df['Date'] = pd.to_datetime(df['Date'])

In [14]:
df['Date'].dtype

dtype('<M8[ns]')

# Use unique() to list the tickers, then nunique() to count tickers, companies and sectors. Why must the number of tickers equal the number of companies?


In [15]:
# Unique will extract the unique instances of a column.
df['Ticker'].unique()

array(['AAPL', 'AMZN', 'F', 'GOOGL', 'META', 'MSFT', 'NFLX', 'NVDA',
       'PLTR', 'TSLA', 'VOO'], dtype=object)

In [16]:
# nunique gives the NUMBER OF UNIQUES
df['Ticker'].nunique()

11

In [17]:
df['Company'].nunique()

11

In [18]:
df['Sector'].nunique()

8

SO EACH TICKER EQUALS TO ONE COMPANY... IF THERE WERE NOT EQUAL THEN THERE WOULD
BE SOMETHING WRONG WITH THE DATAFRAME....

# Sanity check: 11 tickers times about 1,250 trading days should give roughly 13,800 rows. Does len(df) match? If not, use value_counts() on Ticker to find the stock with fewer rows, and suggest a reason (a stock that listed recently would have a shorter history).


In [19]:
len(df)

13783

In [20]:
# len(df) is 13,783, which equals 11 tickers × 1,253 trading days.
# Every ticker has the same number of rows, so none has a shorter history.
df['Ticker'].value_counts()

,count
Ticker,
AAPL,1253
AMZN,1253
F,1253
GOOGL,1253
META,1253
MSFT,1253
NFLX,1253
NVDA,1253
PLTR,1253


# Show 5 random rows with df.sample(5). Pick one row and explain each column in plain words, as if reading a line of a brokerage statement.


In [20]:
df.sample(5)

,Date,Ticker,Company,Sector,Is_ETF,Open,High,Low,Close,Volume,Daily_Return,Up_Day
9078,2023-02-01,NVDA,Nvidia,Semiconductors,False,19.625388,21.121386,19.545654,20.873217,660477000,7.196615,True
8540,2025-12-09,NFLX,Netflix,Media,False,97.029999,97.239998,95.449997,96.709999,51745600,-0.082655,False
7222,2025-09-09,MSFT,Microsoft,Software,False,497.338057,498.151372,493.638515,494.342712,14410500,0.042151,True
2358,2026-04-17,AMZN,Amazon,Retail / Cloud,False,254.990005,256.179993,250.110001,250.559998,52029300,0.344414,True
2245,2025-11-03,AMZN,Amazon,Retail / Cloud,False,255.360001,258.600006,252.899994,254.000000,95997800,4.004586,True


# Part 2: Categorical questions


# You are choosing between sectors and between stocks and a fund, so start by counting groups.
# How many stocks does each sector contain? Use df.groupby('Sector')['Ticker'].nunique(). Then build a small summary DataFrame from it with the dictionary pattern from the course (pd.DataFrame({'stocks': ...})) and give the column an honest name. Which sector holds the most of your picks, and what does that mean for how spread out your money would be?


In [21]:
df.groupby('Sector')['Ticker'].unique()
df.groupby('Sector')['Ticker'].unique()

,Ticker
Sector,
Autos,"[F, TSLA]"
Communication,"[GOOGL, META]"
Consumer Tech,[AAPL]
Index,[VOO]
Media,[NFLX]
Retail / Cloud,[AMZN]
Semiconductors,[NVDA]
Software,"[MSFT, PLTR]"


In [22]:
summary_sectors = pd.DataFrame({'number_of_stocks': df.groupby('Sector')['Ticker'].nunique()})
summary_sectors

,number_of_stocks
Sector,
Autos,2
Communication,2
Consumer Tech,1
Index,1
Media,1
Retail / Cloud,1
Semiconductors,1
Software,2


# What is the mode of Sector (df['Sector'].mode()[0])? Does it describe the sector with the most stocks, or the sector with the most rows? Why can the two differ?


In [23]:
df.head()

,Date,Ticker,Company,Sector,Is_ETF,Open,High,Low,Close,Volume,Daily_Return,Up_Day
1,2021-11-22,AAPL,Apple,Consumer Tech,False,157.808200,161.731936,157.144491,157.164017,117467900,0.292702,True
2,2021-11-23,AAPL,Apple,Consumer Tech,False,157.261660,157.925384,155.250993,157.544724,96041900,0.242235,True
3,2021-11-24,AAPL,Apple,Consumer Tech,False,156.900491,158.257204,155.817072,158.061996,69463600,0.328334,True
4,2021-11-26,AAPL,Apple,Consumer Tech,False,155.748786,156.607703,152.615650,153.054871,76959800,-3.167824,False
5,2021-11-29,AAPL,Apple,Consumer Tech,False,155.553564,157.329988,154.987452,156.402740,88748200,2.187366,True


In [24]:
df['Sector']

,Sector
1,Consumer Tech
2,Consumer Tech
3,Consumer Tech
4,Consumer Tech
5,Consumer Tech
...,...
13789,Index
13790,Index
13791,Index
13792,Index


In [25]:
# WHAT MODE DOES IS THAT IS GOING TO GO INTO THE DF AND IS GOING TO PICK
# THE CONCEPT, OCCURRANCE WITH THE MOST APPEREANCES... SO THE SECTORS
# THAT MOST APPEAR ARE AUTOS, COMMUNICATION, SOFTWARE... IT IS GOING TO
# EVELUATE THE ENTIRITY OF THE ROWS... SO THEY APPEAR THE SAME NUMBER OF TIMES

df['Sector'].mode()

,Sector
0,Autos
1,Communication
2,Software


In [26]:
# BY DOING THE FOLLOWING I AM PICKING THE FIRST SECTOR. THE O,1,3 IS JUST
# ROW LABELS THAT THE SYSTEM IS GIVING...
df['Sector'].mode()[0]

'Autos'

# Up_Day has missing values. Predict what happens if you run df[df['Up_Day']], then run it and compare with your prediction. Fix it in two different ways.


In [27]:
# We are applying a mask. By default, when a mask is applied,
# this mask gives the true ones
df[df['Up_Day']]


,Date,Ticker,Company,Sector,Is_ETF,Open,High,Low,Close,Volume,Daily_Return,Up_Day
1,2021-11-22,AAPL,Apple,Consumer Tech,False,157.808200,161.731936,157.144491,157.164017,117467900,0.292702,True
2,2021-11-23,AAPL,Apple,Consumer Tech,False,157.261660,157.925384,155.250993,157.544724,96041900,0.242235,True
3,2021-11-24,AAPL,Apple,Consumer Tech,False,156.900491,158.257204,155.817072,158.061996,69463600,0.328334,True
5,2021-11-29,AAPL,Apple,Consumer Tech,False,155.553564,157.329988,154.987452,156.402740,88748200,2.187366,True
6,2021-11-30,AAPL,Apple,Consumer Tech,False,156.158714,161.556286,156.090383,161.341553,174048100,3.157753,True
...,...,...,...,...,...,...,...,...,...,...,...,...
13782,2026-11-03,VOO,Vanguard S&P 500 ETF,Index,True,691.880005,695.900024,691.630005,693.539978,6279900,0.321121,True
13783,2026-11-04,VOO,Vanguard S&P 500 ETF,Index,True,701.330017,707.090027,701.090027,704.780029,6941500,1.620678,True
13785,2026-11-06,VOO,Vanguard S&P 500 ETF,Index,True,675.440002,682.250000,675.299988,675.989990,11789200,0.299720,True
13786,2026-11-09,VOO,Vanguard S&P 500 ETF,Index,True,682.000000,694.390015,681.179993,691.869995,7438300,2.349148,True


In [28]:
df[df['Up_Day'] == False]

,Date,Ticker,Company,Sector,Is_ETF,Open,High,Low,Close,Volume,Daily_Return,Up_Day
4,2021-11-26,AAPL,Apple,Consumer Tech,False,155.748786,156.607703,152.615650,153.054871,76959800,-3.167824,False
7,2021-12-01,AAPL,Apple,Consumer Tech,False,163.469344,166.221821,160.589991,160.824249,152052500,-0.320626,False
8,2021-12-02,AAPL,Apple,Consumer Tech,False,154.938650,160.267890,154.021157,159.838425,136739200,-0.612983,False
9,2021-12-03,AAPL,Apple,Consumer Tech,False,160.092220,161.009712,155.895189,157.964417,118023100,-1.172439,False
13,2021-12-09,AAPL,Apple,Consumer Tech,False,170.721462,172.517396,169.755164,170.379837,108923700,-0.296992,False
...,...,...,...,...,...,...,...,...,...,...,...,...
13789,2026-11-12,VOO,Vanguard S&P 500 ETF,Index,True,693.059998,701.419983,692.840027,696.000000,6184800,-0.336504,False
13790,2026-11-13,VOO,Vanguard S&P 500 ETF,Index,True,698.659973,698.809998,687.500000,693.380005,5626200,-0.376436,False
13791,2026-11-16,VOO,Vanguard S&P 500 ETF,Index,True,689.109985,689.219971,678.469971,680.260010,6706700,-1.892180,False
13792,2026-11-17,VOO,Vanguard S&P 500 ETF,Index,True,678.820007,679.979980,668.270020,673.190002,10039700,-1.039310,False


In [30]:
df['Up_Day'].isna().sum()

np.int64(0)

# Use value_counts(normalize=True) on Up_Day. What share of all days were up days? Then compute the share per ticker. Which stock rose on the highest share of days, and which on the lowest?


In [29]:
df['Up_Day'].sample(5)

,Up_Day
7795,False
11038,True
13313,False
1340,True
1290,False


In [30]:
df['Up_Day'].value_counts()

,count
Up_Day,
True,7083
False,6700


In [31]:
# This gives a proporpotion. Multiply by 100 and is the percentage.
df['Up_Day'].value_counts(normalize=True)

,proportion
Up_Day,
True,0.513894
False,0.486106


# Build nvda = df[df['Ticker'] == 'NVDA'] and etf = df[df['Is_ETF'] == True]. How many rows does each have, and which tickers does etf contain?


In [32]:
nvda = df[df['Ticker']=='NVDA']
nvda.shape
nvda

,Date,Ticker,Company,Sector,Is_ETF,Open,High,Low,Close,Volume,Daily_Return,Up_Day
8779,2021-11-22,NVDA,Nvidia,Semiconductors,False,33.370016,34.495062,31.760108,31.815863,754335000,-3.119603,False
8780,2021-11-23,NVDA,Nvidia,Semiconductors,False,31.396706,32.218089,30.744578,31.606781,532163000,-0.657162,False
8781,2021-11-24,NVDA,Nvidia,Semiconductors,False,31.323029,32.710914,30.792365,32.530708,435162000,2.923193,True
8782,2021-11-26,NVDA,Nvidia,Semiconductors,False,32.457035,32.566553,31.212518,31.364847,283069000,-3.583879,False
8783,2021-11-29,NVDA,Nvidia,Semiconductors,False,32.224068,33.265479,31.895513,33.229637,454964000,5.945478,True
...,...,...,...,...,...,...,...,...,...,...,...,...
10027,2026-11-12,NVDA,Nvidia,Semiconductors,False,151.009995,167.139999,148.800003,166.190002,747707600,-4.340064,False
10028,2026-11-13,NVDA,Nvidia,Semiconductors,False,169.910004,170.000000,158.389999,162.850006,365796600,-2.009746,False
10029,2026-11-16,NVDA,Nvidia,Semiconductors,False,159.020004,163.199997,155.009995,156.139999,333822100,-4.120360,False
10030,2026-11-17,NVDA,Nvidia,Semiconductors,False,156.899994,158.190002,150.330002,155.460007,406985100,-0.435502,False


In [33]:
df.info()
etf = df[df['Is_ETF']== True]
etf.shape


<class 'pandas.core.frame.DataFrame'>
Index: 13783 entries, 1 to 13793
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Date          13783 non-null  datetime64[ns]
 1   Ticker        13783 non-null  object        
 2   Company       13783 non-null  object        
 3   Sector        13783 non-null  object        
 4   Is_ETF        13783 non-null  bool          
 5   Open          13783 non-null  float64       
 6   High          13783 non-null  float64       
 7   Low           13783 non-null  float64       
 8   Close         13783 non-null  float64       
 9   Volume        13783 non-null  int64         
 10  Daily_Return  13783 non-null  float64       
 11  Up_Day        13783 non-null  object        
dtypes: bool(1), datetime64[ns](1), float64(5), int64(1), object(4)
memory usage: 1.8+ MB


(1253, 12)

# Write two sentences: is a high share of up days a good reason to pick a stock? Think about what the share ignores (how much it rose on the up days and fell on the down days).


No, a high share of up days is not a good reason on its own, because it ignores how big the gains and losses are. A stock that rises on most days can still lose money if its few down days are much larger than its up days.

# Part 3: What does a typical day look like?
# Prices are not comparable between stocks (one share of Netflix costs far more than one share of Ford), so from here on you work with Daily_Return, which is in percent.


# For the whole of df, calculate the min, max, median, mode and midrange of Daily_Return, following the course notebook. Which ticker and date produced the single best day and the single worst day? (Hint: df.loc[df['Daily_Return'].idxmax()].)


In [34]:
df['Daily_Return'].median() # For median i dont need loc

0.0629069367461987

In [35]:
df['Daily_Return'].min()

-35.11660755113309

In [36]:
# This gives me the row of the lowest return
df['Daily_Return'].idxmin()

np.int64(7627)

In [37]:
# If i want to see what it is in that row i need to use loc.
# The format is: df.loc[row_label, column]
row = df['Daily_Return'].idxmin()
df.loc[row] # This will give me the row label

,7627
Date,2022-04-20 00:00:00
Ticker,NFLX
Company,Netflix
Sector,Media
Is_ETF,False
Open,24.52
High,24.870001
Low,21.250999
Close,22.618999
Volume,1333875000


In [38]:
# This is telling me what ticker was the lowests return ever.
df.loc[row,'Ticker']

'NFLX'

In [39]:
df['Daily_Return'].max()

30.801445797701252

In [40]:
# This tells me what what stock had the highest return ever
rowmax = df['Daily_Return'].idxmax()
df.loc[rowmax,'Ticker']

'PLTR'

In [41]:
df['Daily_Return'].mode() # The most common daily return is 0%

,Daily_Return
0,0.0


NOW I WILL ANSWER: Which ticker and date produced the single best day and the single worst day? (Hint: df.loc[df['Daily_Return'].idxmax()].)

In [73]:
df.head()
df.loc['Ticker']
THIS IS WRONG BECAUSE LOC EXPECTS ROWS AND NOT COLUMNS. LOC expects a row
labeled.

Yes. With one value, loc expects a row label.

The full pattern is df.loc[row, column]. If you give only the first part,
it treats it as the row and returns all columns for that row.

KeyError: 'Ticker'

# idxmax() returns the row label (index) of the largest value in a column. idxmin() returns the row label of the smallest value.

# Compare with max() and min(), which return the value itself:

# this is giving me the row with the largest daily_return

In [42]:
# Yes. loc takes the output of idxmax() (the row label) as its input
# and returns that row.

# Give me the row with the largest daily return:

# THIS IS DISPLYING ALL OF THE INFORMATION OF THE ROW WITH THE HIGHEST RETURN

df.loc[df['Daily_Return'].idxmax()]

,10586
Date,2024-02-06 00:00:00
Ticker,PLTR
Company,Palantir
Sector,Software
Is_ETF,False
Open,20.299999
High,22.18
Low,19.76
Close,21.870001
Volume,420501900


In [43]:
# THIS IS LIKE THE FILTER .....
df['Daily_Return'].idxmin()

np.int64(7627)

In [44]:
# I AM GETTING NOW THE ENTIRE ROW ...
df.loc[df['Daily_Return'].idxmin()]

,7627
Date,2022-04-20 00:00:00
Ticker,NFLX
Company,Netflix
Sector,Media
Is_ETF,False
Open,24.52
High,24.870001
Low,21.250999
Close,22.618999
Volume,1333875000


# Write a function getCentralTendency(ticker) that filters df to one ticker and prints min, max, mean, median, mode and midrange of its Daily_Return. Loop over all 11 tickers.


In [45]:
df['Ticker'].unique()

array(['AAPL', 'AMZN', 'F', 'GOOGL', 'META', 'MSFT', 'NFLX', 'NVDA',
       'PLTR', 'TSLA', 'VOO'], dtype=object)

In [46]:
# I first need to pass the tickers to a variable in the form of a list
tickers_list = df['Ticker'].unique().tolist()
tickers_list

['AAPL',
 'AMZN',
 'F',
 'GOOGL',
 'META',
 'MSFT',
 'NFLX',
 'NVDA',
 'PLTR',
 'TSLA',
 'VOO']

In [49]:
# Creating the function()
def getCentralTendency(ticker):
  var = df[df['Ticker'] == ticker]

  #Basically it passes everything:
  print("This is the var variable:", var)

  '''
  print('Ticker', ticker,'Min', round(var['Daily_Return'].min(),2))
  print('Ticker', ticker, 'Mode',round(var['Daily_Return'].mode()[0],2))
  print('Ticker', ticker, 'Mean',round(var['Daily_Return'].mean(),2))
  print('Ticker', ticker, 'Median',round(var['Daily_Return'].median(),2))
  '''




for i in tickers_list:
  getCentralTendency(i)

NameError: name 'tickers_list' is not defined

In [50]:
# Continuing with the function:
# Creating the function()
def getCentralTendency(ticker):

  #This passes EVERYTHING into the variable
  var = df[df['Ticker'] == ticker]

  # var = df[df['Ticker'] == ticker] keeps only the rows of one ticker.
  # var is a small table (DataFrame) with all the columns, but only for that stock.
  # var['Daily_Return'] picks one column out of that small table.
  # It’s the same as df['Daily_Return'], but only for that ticker,
  # so you get one Series with that stock’s daily returns.

  print('Ticker', ticker,'Min', round(var['Daily_Return'].min(),2))
  print('Ticker', ticker, 'Mode',round(var['Daily_Return'].mode()[0],2))
  print('Ticker', ticker, 'Mean',round(var['Daily_Return'].mean(),2))
  print('Ticker', ticker, 'Median',round(var['Daily_Return'].median(),2))


for i in tickers_list:
  getCentralTendency(i)


NameError: name 'tickers_list' is not defined

# The mode will probably look useless. Explain why (almost every return is a different number), then round the returns to 1 decimal and recalculate. Is the mode more meaningful now?


# Put the mean and the median of every ticker side by side in one DataFrame. For which tickers is the gap largest? Use what you learned about mean versus median: what does a gap say about extreme days?


In [51]:
df[df.groupby('Ticker')['Daily_Return']]

TypeError: unhashable type: 'Series'

# df[df[...]] is only for filtering rows. The inner part must produce True/False values, and the outer df[...] keeps the True rows.

# groupby doesn’t filter anything. It’s a method that you call directly on df: df.groupby(...). It makes piles and calculates on them, so there is no mask and nothing to apply.

# Simple rule: use df[df[...]] when you want to keep only some rows. Don’t use it for groupby.

In [47]:

pd.DataFrame({'Mean':df.groupby('Ticker')['Daily_Return'].median(),
              'Median': df.groupby('Ticker')['Daily_Return'].mean()})

# Yes. Each value in the dictionary is a Series:
# one number per ticker, with the ticker as its index.
# The key ('Mean', 'Median') is just the name that becomes the column header.

,Mean,Median
Ticker,,
AAPL,0.104898,0.080330
AMZN,0.025091,0.056690
F,0.071964,0.012251
GOOGL,0.095195,0.073048
META,0.064777,0.088425
MSFT,0.045539,0.050487
NFLX,-0.024419,0.029712
NVDA,0.224134,0.174589
PLTR,0.052827,0.245981


# ['Daily_Return'] picks which column to work on inside each pile.

# df.groupby('Ticker') makes one pile per ticker, with all columns.
# ['Daily_Return'] keeps only that column in each pile.
# The calculation (.mean(), .median()) comes last.

## So the order is: group by what, then which column, then what calculation. Without the last step you only get the “object” you see in your screenshot.

# Create Dollar_Volume = Close * Volume and compare the median per ticker. Which stock has the most money changing hands on a normal day? Why is Volume alone misleading for this question?


In [53]:
df.head()

,Date,Ticker,Company,Sector,Is_ETF,Open,High,Low,Close,Volume,Daily_Return,Up_Day
1,2021-11-19,AAPL,Apple,Consumer Tech,False,153.874744,157.164053,152.781570,156.705307,117305600,1.697594,True
2,2021-11-22,AAPL,Apple,Consumer Tech,False,157.808231,161.731968,157.144522,157.164047,117467900,0.292741,True
3,2021-11-23,AAPL,Apple,Consumer Tech,False,157.261690,157.925414,155.251023,157.544754,96041900,0.242235,True
4,2021-11-24,AAPL,Apple,Consumer Tech,False,156.900537,158.257250,155.817117,158.062042,69463600,0.328344,True
5,2021-11-26,AAPL,Apple,Consumer Tech,False,155.748802,156.607719,152.615665,153.054886,76959800,-3.167842,False


In [48]:
df['Dollar_Volume'] = df['Close'] * df['Volume']
df['Dollar_Volume']

,Dollar_Volume
1,1.846173e+10
2,1.513089e+10
3,1.097956e+10
4,1.177907e+10
5,1.388046e+10
...,...
13789,4.304621e+09
13790,3.901095e+09
13791,4.562300e+09
13792,6.758626e+09


In [49]:
pd.DataFrame({'Median':df.groupby('Ticker')['Dollar_Volume'].median().sort_values(ascending=False)})

,Median
Ticker,
NVDA,2.442244e+10
TSLA,2.384451e+10
AAPL,1.111494e+10
MSFT,8.530433e+09
AMZN,8.001995e+09
META,6.812357e+09
GOOGL,4.690508e+09
NFLX,2.501649e+09
VOO,2.484184e+09


# In two sentences: if you only look at the median daily return, which stock would you rank first, and what is that ranking missing? (This sets up Part 4.)


It’s two small questions:

Which stock is first? Take the median daily return per ticker and sort high to low, like you just did with Dollar_Volume. The top one is your answer.
# What is the ranking missing? The median only tells you the typical day. It says nothing about risk: how much the returns swing up and down. A stock can have the best typical day and still have huge crashes and spikes.

# Part 4: How risky is each stock?
## Two stocks can have the same median return and very different ride quality. Dispersion measures the ride.



# Calculate the standard deviation of Daily_Return for each ticker (this is the volatility). Rank the tickers from calmest to wildest. Where does VOO, the index fund, land?


In [56]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13783 entries, 1 to 13793
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Date           13783 non-null  datetime64[ns]
 1   Ticker         13783 non-null  object        
 2   Company        13783 non-null  object        
 3   Sector         13783 non-null  object        
 4   Is_ETF         13783 non-null  bool          
 5   Open           13783 non-null  float64       
 6   High           13783 non-null  float64       
 7   Low            13783 non-null  float64       
 8   Close          13783 non-null  float64       
 9   Volume         13783 non-null  int64         
 10  Daily_Return   13783 non-null  float64       
 11  Up_Day         13783 non-null  object        
 12  Dollar_Volume  13783 non-null  float64       
dtypes: bool(1), datetime64[ns](1), float64(6), int64(1), object(4)
memory usage: 1.9+ MB


In [50]:
df.groupby('Ticker')['Daily_Return'].std().sort_values(ascending=True)
# Standard deviation represent volatility. Meaning: How much the daily return
# typical moves from the its average.

,Daily_Return
Ticker,
VOO,1.078804
AAPL,1.771897
MSFT,1.794139
GOOGL,2.059062
AMZN,2.315789
F,2.450353
NFLX,2.765543
META,2.865102
NVDA,3.265723


# Calculate the variance for each ticker and check that it equals the standard deviation squared. In one sentence, why do people quote the standard deviation instead of the variance?


In [51]:
df.groupby('Ticker')['Daily_Return'].var()

var = df.groupby('Ticker')['Daily_Return'].var()
std = df.groupby('Ticker')['Daily_Return'].std()

pd.DataFrame({'Std_squared': std ** 2, 'Variance': var})

,Std_squared,Variance
Ticker,,
AAPL,3.139620,3.139620
AMZN,5.362877,5.362877
F,6.004229,6.004229
GOOGL,4.239735,4.239735
META,8.208811,8.208811
MSFT,3.218936,3.218936
NFLX,7.648226,7.648226
NVDA,10.664948,10.664948
PLTR,17.839313,17.839313


# Write a function getDispersion(ticker) like the one in the course: range, Q1, Q2, Q3, IQR, variance and standard deviation. Loop over all tickers and collect the results in one DataFrame.


In [52]:
# I am grasping on the concepts of the function SO THIS IS MESSY
# In the next code, I will do it the right way, as real life.
# THIS VERSION (THE MESSY) is necessary with the function
# but there is another way
def getDispersion(ticker):
  var = df[df['Ticker'] == ticker]

  print("Ticker", ticker, "Range", var['Daily_Return'].max() - var['Daily_Return'].min())
  print("Ticker", ticker, "Q1", var['Daily_Return'].quantile(0.25))
  print("Ticker", ticker, "Q2", var['Daily_Return'].quantile(0.50))
  print("Ticker", ticker, "Q3", var['Daily_Return'].quantile(0.75))
  print("Ticker", ticker, "IQR",var['Daily_Return'].quantile(0.75)-var['Daily_Return'].quantile(.25))



tickers_list = df['Ticker'].unique().tolist()
for i in tickers_list:
  getDispersion(i)

Ticker AAPL Range 24.574508207192636
Ticker AAPL Q1 -0.8023312208023659
Ticker AAPL Q2 0.1048978007099687
Ticker AAPL Q3 0.9719800358061416
Ticker AAPL IQR 1.7743112566085073
Ticker AMZN Range 29.37002655295551
Ticker AMZN Q1 -1.226272026117703
Ticker AMZN Q2 0.0250905983453009
Ticker AMZN Q3 1.344896590822775
Ticker AMZN IQR 2.571168616940478
Ticker F Range 31.53901931248172
Ticker F Q1 -1.3029307255749467
Ticker F Q2 0.0719638718917226
Ticker F Q3 1.2779606299160085
Ticker F IQR 2.580891355490955
Ticker GOOGL Range 22.865550548300753
Ticker GOOGL Q1 -1.0881373979536924
Ticker GOOGL Q2 0.0951953315151188
Ticker GOOGL Q3 1.1630362925658
Ticker GOOGL IQR 2.2511736905194923
Ticker META Range 49.672523099767815
Ticker META Q1 -1.2182953839471875
Ticker META Q2 0.0647765381241116
Ticker META Q3 1.386836207740627
Ticker META IQR 2.6051315916878144
Ticker MSFT Range 25.49988505452128
Ticker MSFT Q1 -0.8937212670254824
Ticker MSFT Q2 0.0455389938294636
Ticker MSFT Q3 1.0003229628053223
Ticker

In [53]:
# This dataFrame has all of the tickers with all of their daily returns
# I can see 1253 Daily Returns
g = df.groupby('Ticker')['Daily_Return']
g.size()

,Daily_Return
Ticker,
AAPL,1253
AMZN,1253
F,1253
GOOGL,1253
META,1253
MSFT,1253
NFLX,1253
NVDA,1253
PLTR,1253


In [54]:
# NEXT
q1 = g.quantile(0.25)
q3 = g.quantile(0.75)
q3

,Daily_Return
Ticker,
AAPL,0.971980
AMZN,1.344897
F,1.277961
GOOGL,1.163036
META,1.386836
MSFT,1.000323
NFLX,1.332384
NVDA,1.988551
PLTR,2.186444


In [55]:
# If i want to display the two series next to each other
pd.DataFrame({'Q1':q1,'Q3':q3})

,Q1,Q3
Ticker,,
AAPL,-0.802331,0.971980
AMZN,-1.226272,1.344897
F,-1.302931,1.277961
GOOGL,-1.088137,1.163036
META,-1.218295,1.386836
MSFT,-0.893721,1.000323
NFLX,-1.228940,1.332384
NVDA,-1.709509,1.988551
PLTR,-1.948052,2.186444


In [56]:
# Creating all the Data in one DataFrame
# The ticker part comes from the g. dataframe where I GROUPED BY previously

dispersion = pd.DataFrame({
    'Range': g.max()-g.min(),
    'Q1': q1,
    'Q2': g.median(),
    'Q3': q3,
    'IQR': q3-q1
}).round(2)

dispersion

,Range,Q1,Q2,Q3,IQR
Ticker,,,,,
AAPL,24.57,-0.80,0.10,0.97,1.77
AMZN,29.37,-1.23,0.03,1.34,2.57
F,31.54,-1.30,0.07,1.28,2.58
GOOGL,22.87,-1.09,0.10,1.16,2.25
META,49.67,-1.22,0.06,1.39,2.61
MSFT,25.50,-0.89,0.05,1.00,1.89
NFLX,51.17,-1.23,-0.02,1.33,2.56
NVDA,41.34,-1.71,0.22,1.99,3.70
PLTR,52.11,-1.95,0.05,2.19,4.13


# Find the ticker with the largest range relative to its IQR. What does a wide range with a small IQR tell you about how the extreme days behave?


In [57]:
dispersion['Range_to_IQR'] = dispersion['Range'] / dispersion['IQR']
dispersion['Range_to_IQR'].sort_values(ascending=False)

,Range_to_IQR
Ticker,
NFLX,19.988281
META,19.030651
AAPL,13.881356
VOO,13.576577
MSFT,13.492063
PLTR,12.617433
F,12.224806
AMZN,11.428016
NVDA,11.172973


# Count the days with a drop of more than 5% (Daily_Return < -5) per ticker, using a mask and value_counts(). Which stock gave you the most days like that? How many of those days did VOO have?


In [118]:
df.head()

,Date,Ticker,Company,Sector,Is_ETF,Open,High,Low,Close,Volume,Daily_Return,Up_Day,Dollar_Volume
1,2021-11-19,AAPL,Apple,Consumer Tech,False,153.874744,157.164053,152.781570,156.705307,117305600,1.697594,True,1.838241e+10
2,2021-11-22,AAPL,Apple,Consumer Tech,False,157.808231,161.731968,157.144522,157.164047,117467900,0.292741,True,1.846173e+10
3,2021-11-23,AAPL,Apple,Consumer Tech,False,157.261690,157.925414,155.251023,157.544754,96041900,0.242235,True,1.513090e+10
4,2021-11-24,AAPL,Apple,Consumer Tech,False,156.900537,158.257250,155.817117,158.062042,69463600,0.328344,True,1.097956e+10
5,2021-11-26,AAPL,Apple,Consumer Tech,False,155.748802,156.607719,152.615665,153.054886,76959800,-3.167842,False,1.177907e+10


In [58]:
df[df['Daily_Return'] < -5]

,Date,Ticker,Company,Sector,Is_ETF,Open,High,Low,Close,Volume,Daily_Return,Up_Day,Dollar_Volume
114,2022-05-05,AAPL,Apple,Consumer Tech,False,160.130062,160.354836,151.432113,153.210800,130525300,-5.571600,False,1.999789e+10
118,2022-05-11,AAPL,Apple,Consumer Tech,False,150.235435,152.143961,142.708980,143.384308,142689800,-5.184139,False,2.045948e+10
123,2022-05-18,AAPL,Apple,Consumer Tech,False,143.726859,144.226007,136.924657,137.825104,109742900,-5.641925,False,1.512533e+10
203,2022-09-13,AAPL,Apple,Consumer Tech,False,156.716699,157.343957,150.316699,150.777344,122656600,-5.867963,False,1.849384e+10
844,2025-04-03,AAPL,Apple,Consumer Tech,False,204.128361,206.064981,199.867831,201.794510,103419000,-9.245629,False,2.086939e+10
...,...,...,...,...,...,...,...,...,...,...,...,...,...
12442,2026-07-02,TSLA,Tesla,Autos,False,428.010010,432.350006,389.299988,393.450012,73717700,-7.488826,False,2.900423e+10
12456,2026-07-23,TSLA,Tesla,Autos,False,341.000000,342.109985,315.739990,319.690002,115606400,-14.523677,False,3.695821e+10
12487,2026-09-04,TSLA,Tesla,Autos,False,362.070007,364.690002,351.320007,354.079987,65018200,-5.922366,False,2.302164e+10
12530,2026-11-05,TSLA,Tesla,Autos,False,374.119995,374.299988,338.149994,343.690002,79675500,-10.773904,False,2.738367e+10


In [59]:
# SO IT IS ALL ABOUT ROWS/// COUNTING THE ROWS
minus_five = df[df['Daily_Return'] < -5]
minus_five['Ticker'].value_counts()

,count
Ticker,
PLTR,103
TSLA,92
NVDA,61
META,33
NFLX,30
F,28
AMZN,22
GOOGL,12
AAPL,8


# Stretch: multiply each standard deviation by the square root of 252 (trading days in a year) to get an approximate yearly volatility. If a stock has 50% yearly volatility, what does that mean for a year in which you hold it? Answer in plain words, without a formula.


In [66]:
# Calculating the standard deviation
Tickers_Standard_Deviation = df.groupby('Ticker')['Daily_Return'].std() * np.sqrt(252)
Tickers_Standard_Deviation


,Daily_Return
Ticker,
AAPL,28.127997
AMZN,36.762004
F,38.898145
GOOGL,32.686590
META,45.482087
MSFT,28.481078
NFLX,43.901629
NVDA,51.841749
PLTR,67.048542


# It’s a convention, not a law, and it does rest on an assumption.

# Volatility = standard deviation of returns. This is the standard definition in finance. It’s a choice: measuring risk as “how much returns spread around their average”.
# × √252 comes from a formula. If daily returns are independent of each other, variances add up over time: the variance over 252 days is 252 × the daily variance. Taking the square root gives std × √252.

# In two sentences: which stock would you call the riskiest and which the safest, and which number did you base that on?


In [ ]:
'''
PLTR is the riskiest (the highest, about 67%) and VOO is the safest
(the lowest, about 17%).
'''

# Part 5: Do they move together?
### Owning 10 stocks only spreads your risk if they do not all fall on the same days. Correlation measures that.


## Reshape the data so each ticker is a column: returns = df.pivot(index='Date', columns='Ticker', values='Daily_Return'). What is its shape, and why is the first row mostly empty?


In [69]:
returns = df.pivot(index='Date', columns='Ticker', values='Daily_Return')
returns

Ticker,AAPL,AMZN,F,GOOGL,META,MSFT,NFLX,NVDA,PLTR,TSLA,VOO
Date,,,,,,,,,,,
2021-11-22,0.292702,-2.828730,5.621444,-1.762259,-1.242388,-0.955978,-2.887447,-3.119603,-3.549744,1.742212,-0.264450
2021-11-23,0.242235,0.209096,-1.367169,-0.355434,-1.102608,-0.632645,-0.779733,-0.657162,-0.290554,-4.135293,0.130261
2021-11-24,0.328334,0.010332,0.247515,0.231849,1.129747,0.068109,0.646736,2.923193,2.525500,0.628477,0.246228
2021-11-26,-3.167824,-2.118470,-2.469152,-2.694359,-2.328066,-2.435584,1.116530,-3.583879,-0.378967,-3.053760,-2.175979
2021-11-29,2.187366,1.626743,-0.405072,2.354372,1.473976,2.108145,-0.270417,5.945478,0.190200,5.090023,1.208138
...,...,...,...,...,...,...,...,...,...,...,...
2026-11-12,-0.135069,0.477135,-2.857146,1.789061,-0.820655,0.358835,-2.866443,-4.340064,-0.475633,1.691233,-0.336504
2026-11-13,0.391677,-0.291654,0.084036,-0.980925,2.899251,0.168261,-0.861377,-2.009746,-2.092316,-1.819099,-0.376436
2026-11-16,-0.979591,-3.341335,-2.854745,-4.074108,-2.381217,-1.124306,-2.252617,-4.120360,-4.524082,-2.685667,-1.892180


In [70]:
returns.shape

(1253, 11)

## Run returns.corr(). Which two stocks are the most correlated and which two the least? Which stock moves least with VOO?


In [71]:
returns.corr()

Ticker,AAPL,AMZN,F,GOOGL,META,MSFT,NFLX,NVDA,PLTR,TSLA,VOO
Ticker,,,,,,,,,,,
AAPL,1.000000,0.481047,0.369343,0.501178,0.433386,0.527303,0.371920,0.461009,0.364965,0.464524,0.708698
AMZN,0.481047,1.000000,0.313296,0.621873,0.576919,0.612130,0.441930,0.511782,0.447427,0.447734,0.701767
F,0.369343,0.313296,1.000000,0.308714,0.280126,0.282419,0.225109,0.314205,0.271816,0.351807,0.537580
GOOGL,0.501178,0.621873,0.308714,1.000000,0.515860,0.532632,0.351385,0.486338,0.374342,0.419711,0.663025
META,0.433386,0.576919,0.280126,0.515860,1.000000,0.507537,0.417910,0.477452,0.386265,0.351373,0.606211
MSFT,0.527303,0.612130,0.282419,0.532632,0.507537,1.000000,0.401618,0.552044,0.431652,0.387143,0.691964
NFLX,0.371920,0.441930,0.225109,0.351385,0.417910,0.401618,1.000000,0.387418,0.389092,0.308853,0.468155
NVDA,0.461009,0.511782,0.314205,0.486338,0.477452,0.552044,0.387418,1.000000,0.475612,0.475329,0.710673
PLTR,0.364965,0.447427,0.271816,0.374342,0.386265,0.431652,0.389092,0.475612,1.000000,0.467352,0.543600


## Explain in plain words what a correlation of 0.2 versus 0.8 means for someone who owns both stocks.:

With 0.8 the two stocks mostly rise and fall together, so owning both gives you little protection: when one has a bad day, the other probably does too. With 0.2 they move mostly independently, so a bad day for one is often cushioned by the other, and owning both makes your total result smoother.


## Pitfall test: calculate the correlation of the Close prices column (closing price) of NVDA and AAPL, then the correlation of their Daily_Return. Compare the two numbers. Which one tells you whether they fall on the same days, and why do prices trending upward distort the other one?


In [72]:
df.head()

,Date,Ticker,Company,Sector,Is_ETF,Open,High,Low,Close,Volume,Daily_Return,Up_Day,Dollar_Volume
1,2021-11-22,AAPL,Apple,Consumer Tech,False,157.808200,161.731936,157.144491,157.164017,117467900,0.292702,True,1.846173e+10
2,2021-11-23,AAPL,Apple,Consumer Tech,False,157.261660,157.925384,155.250993,157.544724,96041900,0.242235,True,1.513089e+10
3,2021-11-24,AAPL,Apple,Consumer Tech,False,156.900491,158.257204,155.817072,158.061996,69463600,0.328334,True,1.097956e+10
4,2021-11-26,AAPL,Apple,Consumer Tech,False,155.748786,156.607703,152.615650,153.054871,76959800,-3.167824,False,1.177907e+10
5,2021-11-29,AAPL,Apple,Consumer Tech,False,155.553564,157.329988,154.987452,156.402740,88748200,2.187366,True,1.388046e+10


In [73]:
returns

Ticker,AAPL,AMZN,F,GOOGL,META,MSFT,NFLX,NVDA,PLTR,TSLA,VOO
Date,,,,,,,,,,,
2021-11-22,0.292702,-2.828730,5.621444,-1.762259,-1.242388,-0.955978,-2.887447,-3.119603,-3.549744,1.742212,-0.264450
2021-11-23,0.242235,0.209096,-1.367169,-0.355434,-1.102608,-0.632645,-0.779733,-0.657162,-0.290554,-4.135293,0.130261
2021-11-24,0.328334,0.010332,0.247515,0.231849,1.129747,0.068109,0.646736,2.923193,2.525500,0.628477,0.246228
2021-11-26,-3.167824,-2.118470,-2.469152,-2.694359,-2.328066,-2.435584,1.116530,-3.583879,-0.378967,-3.053760,-2.175979
2021-11-29,2.187366,1.626743,-0.405072,2.354372,1.473976,2.108145,-0.270417,5.945478,0.190200,5.090023,1.208138
...,...,...,...,...,...,...,...,...,...,...,...
2026-11-12,-0.135069,0.477135,-2.857146,1.789061,-0.820655,0.358835,-2.866443,-4.340064,-0.475633,1.691233,-0.336504
2026-11-13,0.391677,-0.291654,0.084036,-0.980925,2.899251,0.168261,-0.861377,-2.009746,-2.092316,-1.819099,-0.376436
2026-11-16,-0.979591,-3.341335,-2.854745,-4.074108,-2.381217,-1.124306,-2.252617,-4.120360,-4.524082,-2.685667,-1.892180


In [78]:
prices = df.pivot(index='Date', columns='Ticker', values='Close')
prices

Ticker,AAPL,AMZN,F,GOOGL,META,MSFT,NFLX,NVDA,PLTR,TSLA,VOO
Date,,,,,,,,,,,
2021-11-22,157.164017,178.628494,14.913496,144.922974,337.774902,326.740753,65.919998,31.815863,20.650000,385.623322,400.784454
2021-11-23,157.544724,179.001999,14.709603,144.407867,334.050568,324.673645,65.405998,31.606781,20.590000,369.676666,401.306519
2021-11-24,158.061996,179.020493,14.746012,144.742676,337.824493,324.894775,65.829002,32.530708,21.110001,372.000000,402.294647
2021-11-26,153.054871,175.227997,14.381910,140.842789,329.959717,316.981689,66.564003,31.364847,21.030001,360.640015,393.540802
2021-11-29,156.402740,178.078506,14.323653,144.158752,334.823242,323.664124,66.384003,33.229637,21.070000,378.996674,398.295319
...,...,...,...,...,...,...,...,...,...,...,...
2026-11-12,354.880005,267.440002,11.900000,298.700012,623.609985,523.000000,62.689999,166.190002,171.580002,397.450012,696.000000
2026-11-13,356.269989,266.660004,11.910000,295.769989,641.690002,523.880005,62.150002,162.850006,167.990005,390.220001,693.380005
2026-11-16,352.779999,257.750000,11.570000,283.720001,626.409973,517.989990,60.750000,156.139999,160.389999,379.739990,680.260010


In [79]:
# Step 2: correlate the two price columns. Pick the two columns from prices
# and call .corr() on one with the other as the argument:

prices['NVDA'].corr(prices['AAPL'])

np.float64(0.9260711800839293)

## Step 3: do the same with the daily returns.

In [80]:
returns

Ticker,AAPL,AMZN,F,GOOGL,META,MSFT,NFLX,NVDA,PLTR,TSLA,VOO
Date,,,,,,,,,,,
2021-11-22,0.292702,-2.828730,5.621444,-1.762259,-1.242388,-0.955978,-2.887447,-3.119603,-3.549744,1.742212,-0.264450
2021-11-23,0.242235,0.209096,-1.367169,-0.355434,-1.102608,-0.632645,-0.779733,-0.657162,-0.290554,-4.135293,0.130261
2021-11-24,0.328334,0.010332,0.247515,0.231849,1.129747,0.068109,0.646736,2.923193,2.525500,0.628477,0.246228
2021-11-26,-3.167824,-2.118470,-2.469152,-2.694359,-2.328066,-2.435584,1.116530,-3.583879,-0.378967,-3.053760,-2.175979
2021-11-29,2.187366,1.626743,-0.405072,2.354372,1.473976,2.108145,-0.270417,5.945478,0.190200,5.090023,1.208138
...,...,...,...,...,...,...,...,...,...,...,...
2026-11-12,-0.135069,0.477135,-2.857146,1.789061,-0.820655,0.358835,-2.866443,-4.340064,-0.475633,1.691233,-0.336504
2026-11-13,0.391677,-0.291654,0.084036,-0.980925,2.899251,0.168261,-0.861377,-2.009746,-2.092316,-1.819099,-0.376436
2026-11-16,-0.979591,-3.341335,-2.854745,-4.074108,-2.381217,-1.124306,-2.252617,-4.120360,-4.524082,-2.685667,-1.892180


In [82]:
returns['NVDA'].corr(returns['AAPL'])

np.float64(0.46100928893568444)

## 0.461, which matches the value in the correlation matrix.

## comparing the two:

## Prices: 0.926
## Returns: 0.461

## The price correlation is almost double, but it mostly reflects that both prices climbed over five years. The return correlation, 0.461, is the one that tells  whether they fall on the same days: moderately, but far from always.

## So the answer to the exercise: the return correlation tells  whether they fall on the same days. Prices trending upward make unrelated stocks look linked, which inflates the other number.

#Check the matrix: what should the diagonal be, and what range must every value fall in? If something is outside, you have a bug.


In [83]:
returns.corr()

Ticker,AAPL,AMZN,F,GOOGL,META,MSFT,NFLX,NVDA,PLTR,TSLA,VOO
Ticker,,,,,,,,,,,
AAPL,1.000000,0.481047,0.369343,0.501178,0.433386,0.527303,0.371920,0.461009,0.364965,0.464524,0.708698
AMZN,0.481047,1.000000,0.313296,0.621873,0.576919,0.612130,0.441930,0.511782,0.447427,0.447734,0.701767
F,0.369343,0.313296,1.000000,0.308714,0.280126,0.282419,0.225109,0.314205,0.271816,0.351807,0.537580
GOOGL,0.501178,0.621873,0.308714,1.000000,0.515860,0.532632,0.351385,0.486338,0.374342,0.419711,0.663025
META,0.433386,0.576919,0.280126,0.515860,1.000000,0.507537,0.417910,0.477452,0.386265,0.351373,0.606211
MSFT,0.527303,0.612130,0.282419,0.532632,0.507537,1.000000,0.401618,0.552044,0.431652,0.387143,0.691964
NFLX,0.371920,0.441930,0.225109,0.351385,0.417910,0.401618,1.000000,0.387418,0.389092,0.308853,0.468155
NVDA,0.461009,0.511782,0.314205,0.486338,0.477452,0.552044,0.387418,1.000000,0.475612,0.475329,0.710673
PLTR,0.364965,0.447427,0.271816,0.374342,0.386265,0.431652,0.389092,0.475612,1.000000,0.467352,0.543600


## The check:

## Diagonal: every value must be exactly 1.0, because each stock is perfectly correlated with itself.
## Range: every value must be between -1 and 1. A value outside that range means a bug.

## Look at the table: the diagonal is all 1.000000 and the other values are between 0.22 and 0.71, so it passes.